# ALPR Plat Nomor Indonesia — YOLO26n + PaddleOCR
1. **Dataset plat** (format YOLO)
2. **Fine-tune YOLO26n** dari bobot pretrained COCO → detektor plat
3. **Baca teks plat**: YOLO memotong plat → PaddleOCR `en_PP-OCRv5_mobile_rec` (pretrained) membaca → `fix_plate` merapikan

YOLO26 bersifat **end-to-end (tanpa NMS)**: model langsung mengeluarkan maksimal 300 kotak final,
sehingga lebih cepat di CPU Mini PC dibanding YOLOv8n. Butuh `ultralytics>=8.4`.

**Runtime: GPU (T4/L4).** YOLO memakai GPU, PaddleOCR memakai CPU sehingga tidak bentrok dengan PyTorch.
Langkah yang hasilnya sudah ada di Drive otomatis dilewati. Hasil YOLO26n disimpan terpisah dari YOLOv8n
(`yolo26n_plate_best.pt`, `models/minipc_yolo26n/`), jadi model lama tidak tertimpa.


## 0. Pengaturan

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT   = '/content/drive/MyDrive/0 - KEPSTON/alpr_new'
DET    = f'{ROOT}/data/detect'      # dataset plat format YOLO: train/valid/test
MODELS = f'{ROOT}/models'
RETRAIN_YOLO = False                # True = latih ulang walaupun model sudah ada

import os
for p in [DET, MODELS, '/content/code']:
    os.makedirs(p, exist_ok=True)
!pip install -q -U "ultralytics>=8.4.0" roboflow   # YOLO26 mulai ultralytics 8.4
!pip install -q paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!pip install -q paddleocr==3.5.0 langchain-text-splitters   # paddleocr ikut meng-import langchain_text_splitters, tetapi tidak memasangnya

In [ ]:
%%writefile /content/code/plate_utils.py
"""Utilitas plat nomor Indonesia: pembersihan, koreksi karakter per posisi, validasi pola."""
import re

LETTER_FIX = {'0': 'O', '1': 'I', '2': 'Z', '4': 'A', '5': 'S', '6': 'G', '7': 'T', '8': 'B'}
DIGIT_FIX = {'O': '0', 'D': '0', 'Q': '0', 'U': '0', 'I': '1', 'L': '1', 'J': '1',
             'Z': '2', 'A': '4', 'S': '5', 'G': '6', 'T': '7', 'B': '8'}
PATTERN = re.compile(r'^([A-Z]{1,2})([0-9]{1,4})([A-Z]{0,3})$')

# Kode wilayah yang umum dipakai. Dipakai sebagai penalti lunak (bukan penolakan),
# sehingga kode yang tidak ada di daftar tetap bisa diterima bila tidak ada pilihan lain.
REGION_CODES = set('A B D E F G H K L M N P R S T W Z'.split()) | set(
    'AA AB AD AE AG BA BB BD BE BG BH BK BL BM BN BP DA DB DC DD DE DG DH DK DL DM DN DR DS DT '
    'EA EB ED KB KH KT KU PA PB'.split())


SLASHED_O = str.maketrans({'Ø': 'O', 'ø': 'O'})   # pada plat, O bergaris miring (Ø) = huruf O


def clean(text):
    """Huruf kapital, O bergaris miring (Ø) -> huruf O, buang semua selain A-Z dan 0-9."""
    return re.sub(r'[^A-Z0-9]', '', str(text).translate(SLASHED_O).upper())


def crop_main_line(crop):
    """Ambil baris nomor utama; buang baris masa berlaku (MM.YY) HANYA bila terpisah jelas.
    Bila ragu (baris menempel, plat miring, gelap), potongan dikembalikan utuh.
    Dipakai SAMA PERSIS saat membuat dataset OCR (notebook 03) dan di pipeline (notebook 05 / mini PC)."""
    import cv2
    import numpy as np
    h, w = crop.shape[:2]
    if h < 16 or w < 16:
        return crop
    x0, x1 = int(w * 0.08), int(w * 0.92)
    g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
    g = g[:, x0:x1]
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if b.mean() > 127:                       # plat putih teks hitam -> balik agar teks = putih
        b = 255 - b
    prof = (b > 0).mean(axis=1)
    prof[prof > 0.80] = 0                    # garis bingkai plat (hampir penuh) bukan teks
    on = prof > max(0.04, prof.max() * 0.15)
    bands, start = [], None
    for y, v in enumerate(np.append(on, False)):
        if v and start is None:
            start = y
        elif not v and start is not None:
            if y - start >= 2:
                bands.append((start, y))
            start = None
    if len(bands) < 2:
        return crop
    main = max(bands, key=lambda t: t[1] - t[0])
    mh = main[1] - main[0]
    below = [t for t in bands if t[0] >= main[1]]
    if mh < h * 0.22 or not below:
        return crop
    date = below[0]
    if (date[1] - date[0]) > mh * 0.75:      # pita di bawah sama besar -> bukan baris tanggal
        return crop
    cut = (main[1] + date[0]) // 2           # potong di tengah celah
    top = max(0, main[0] - int(mh * 0.25))
    return crop[top:cut]


def fix_plate(text):
    """Cari pembagian [huruf 1-2][angka 1-4][huruf 0-3] dengan koreksi karakter paling sedikit.
    Mengembalikan string tanpa spasi (mis. 'B1234XYZ') atau None bila tidak bisa dibentuk."""
    s = clean(text)
    if len(s) > 9 and s[-4:].isdigit():        # sisa baris masa berlaku (MM.YY) ikut terbaca
        s = s[:-4]
    best = None
    for a in (1, 2):
        for b in range(1, 5):
            c = len(s) - a - b
            if c < 0 or c > 3:
                continue
            out, cost, ok = [], 0, True
            for i, ch in enumerate(s):
                if i < a or i >= a + b:          # posisi huruf
                    if ch.isalpha():
                        out.append(ch)
                    elif ch in LETTER_FIX:
                        out.append(LETTER_FIX[ch]); cost += 1
                    else:
                        ok = False; break
                else:                             # posisi angka
                    if ch.isdigit():
                        out.append(ch)
                    elif ch in DIGIT_FIX:
                        out.append(DIGIT_FIX[ch]); cost += 1
                    else:
                        ok = False; break
            if ok:
                if ''.join(out[:a]) not in REGION_CODES:
                    cost += 2                     # kode wilayah tidak dikenal
                key = (cost, -b)
                if best is None or key < best[0]:
                    best = (key, ''.join(out))
    return best[1] if best else None


def is_valid(plate):
    return bool(plate) and bool(PATTERN.match(plate))


def format_plate(plate):
    """'B1234XYZ' -> 'B 1234 XYZ'"""
    m = PATTERN.match(plate or '')
    return ' '.join(g for g in m.groups() if g) if m else (plate or '')


def levenshtein(a, b):
    a, b = a or '', b or ''
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


# Karakter yang sering tertukar oleh OCR pada plat; satu grup dianggap sama saat dicocokkan ke database.
CONFUSABLE = ['0OQD', '1IL', '2Z', '4A', '5S', '6G', '7T', '8B']
_CANON = {ch: grp[0] for grp in CONFUSABLE for ch in grp}


def canon(text):
    """Bentuk kanonik untuk pencocokan: 'BM6408QO' dan 'BM640800' -> '8M640800'."""
    return ''.join(_CANON.get(ch, ch) for ch in clean(text))


def match_plate(text, registered, max_edit=1):
    """Cocokkan hasil OCR ke daftar plat terdaftar.
    Urutan: SAMA PERSIS -> MIRIP (beda hanya karakter yang mudah tertukar) -> EDIT (selisih <= max_edit karakter).
    Hasil: {'plat': plat terdaftar atau None, 'cara': 'EXACT'|'MIRIP'|'EDIT'|'AMBIGU'|'TIDAK ADA', 'kandidat': [...]}
    'AMBIGU' = lebih dari satu plat cocok; biarkan RFID yang menentukan dari daftar kandidat."""
    reg = {clean(p): p for p in registered}
    reads = [r for r in (fix_plate(text), clean(text)) if r]
    for r in reads:
        if r in reg:
            return {'plat': reg[r], 'cara': 'EXACT', 'kandidat': [reg[r]]}
    keys = {canon(r) for r in reads}
    for cara, cocok in (('MIRIP', lambda k: canon(k) in keys),
                        ('EDIT', lambda k: min(levenshtein(canon(k), c) for c in keys) <= max_edit)):
        cand = [reg[k] for k in reg if keys and cocok(k)]
        if len(cand) == 1:
            return {'plat': cand[0], 'cara': cara, 'kandidat': cand}
        if cand:
            return {'plat': None, 'cara': 'AMBIGU', 'kandidat': cand}
    return {'plat': None, 'cara': 'TIDAK ADA', 'kandidat': []}


In [ ]:
import sys, shutil, glob, random
from pathlib import Path
import cv2, numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '/content/code')
import importlib, plate_utils
importlib.reload(plate_utils)   # pakai versi terbaru bila sel 4 dijalankan ulang di sesi yang sama
from plate_utils import fix_plate, is_valid, format_plate, crop_main_line, match_plate
print(fix_plate('b 1234 xyz'), fix_plate('BI234XYZ 05.28'), fix_plate('B 123 ØK'))   # B1234XYZ B1234XYZ B123OK

## 1. Dataset plat
Dilewati bila `data/detect/train` sudah berisi gambar. Bila belum, diunduh dari Roboflow (format YOLOv8),
semua kelas dijadikan satu kelas `plate`. Dataset lalu disalin ke disk lokal Colab agar training cepat.

In [ ]:
DATASETS = [  # (workspace, project, versi atau None = terbaru)
    ('samuel-nathaniel-h-f0y9g', 'indonesian-license-plates', None),
    ('ksp-workspace', 'indonesia-license-plate-iqrtj', None),
    ('ali-mustofa-cxrgi', 'license-plate-indonesia', None),
]
n_img = lambda s: len(list((Path(DET) / s / 'images').glob('*')))


def roboflow_key():
    """Minta API key Roboflow diketik manual (tersembunyi); diulang bila kosong, maksimal 3 kali."""
    from getpass import getpass
    for _ in range(3):
        key = getpass('Roboflow API key (Private API Key, lalu ENTER): ').strip()
        if key:
            return key
        print('API key kosong, ketik ulang.')
    raise ValueError('API key Roboflow tidak diisi.')


if not n_img('train'):
    from roboflow import Roboflow
    rf = Roboflow(api_key=roboflow_key())
    for ws, proj, ver in DATASETS:
        p = rf.workspace(ws).project(proj)
        ver = ver or max(int(str(v.version).split('/')[-1]) for v in p.versions())
        src = Path(p.version(ver).download('yolov8', location=f'/content/rf/{proj}').location)
        for split in ['train', 'valid', 'test']:
            if not (src / split / 'images').exists():
                continue
            for sub in ['images', 'labels']:
                (Path(DET) / split / sub).mkdir(parents=True, exist_ok=True)
            for img in (src / split / 'images').glob('*'):
                lbl = src / split / 'labels' / f'{img.stem}.txt'
                rows = []
                for ln in (lbl.read_text().splitlines() if lbl.exists() else []):
                    v = ln.split()
                    if len(v) == 5:
                        cx, cy, w, h = (min(1.0, max(0.0, float(t))) for t in v[1:])
                        if w > 0.002 and h > 0.002:
                            rows.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
                if rows:
                    shutil.copy(img, Path(DET) / split / 'images' / f'{proj}__{img.name}')
                    (Path(DET) / split / 'labels' / f'{proj}__{img.stem}.txt').write_text('\n'.join(rows))
print('Dataset:', {s: n_img(s) for s in ['train', 'valid', 'test']})

# Salin ke disk lokal Colab; salin ulang bila salinan lokal tidak ada / tidak lengkap
# (mis. penyalinan sebelumnya terputus sehingga hanya ada folder train).
LOCAL = Path('/content/detect')
n_local = lambda s: len(list((LOCAL / s / 'images').glob('*')))
if any(n_local(s) != n_img(s) for s in ['train', 'valid', 'test']):
    shutil.rmtree(LOCAL, ignore_errors=True)
    shutil.copytree(DET, LOCAL)
print('Salinan lokal:', {s: n_local(s) for s in ['train', 'valid', 'test']})
Path('/content/plate.yaml').write_text(
    'path: /content/detect\ntrain: train/images\nval: valid/images\ntest: test/images\nnames:\n  0: plate\n')

## 2. Fine-tune YOLO26n
Mulai dari `yolo26n.pt` (pretrained COCO), dilatih ulang untuk satu kelas `plate`:
640 px, 100 epoch, batch 16, berhenti otomatis bila 20 epoch tidak membaik.
Optimizer dibiarkan `auto` (Ultralytics memilih MuSGD untuk YOLO26).
Bila sesi terputus di tengah training, jalankan sel ini lagi: training **dilanjutkan** dari `last.pt`.


In [ ]:
from ultralytics import YOLO
BEST = f'{MODELS}/yolo26n_plate_best.pt'
RUN  = Path(f'{MODELS}/runs/yolo26n_plate')

if RETRAIN_YOLO or not Path(BEST).exists():
    last = RUN / 'weights' / 'last.pt'
    if last.exists() and not RETRAIN_YOLO:
        YOLO(str(last)).train(resume=True)
    else:
        YOLO('yolo26n.pt').train(data='/content/plate.yaml', imgsz=640, epochs=100, batch=16, patience=20,
                                 seed=42, project=str(RUN.parent), name=RUN.name, exist_ok=True)
    shutil.copy(RUN / 'weights' / 'best.pt', BEST)

det = YOLO(BEST)
m = det.val(data='/content/plate.yaml', split='test', imgsz=640, verbose=False, plots=False)
print(f'Test | mAP50 = {m.box.map50:.3f} | mAP50-95 = {m.box.map:.3f} | precision = {m.box.mp:.3f} | recall = {m.box.mr:.3f}')

if (RUN / 'results.png').exists():
    plt.figure(figsize=(16, 8)); plt.imshow(plt.imread(RUN / 'results.png')); plt.axis('off'); plt.show()

## 3. Baca teks plat dengan PaddleOCR
Untuk setiap plat yang terdeteksi: potong (dengan sedikit margin) → buang baris masa berlaku bila terpisah jelas →
PaddleOCR membaca → `fix_plate` merapikan sesuai pola plat Indonesia (`B 1234 XYZ`; Ø dibaca huruf O).
Hijau = sesuai pola plat, merah = tidak sesuai.

In [ ]:
from paddleocr import TextRecognition
# 'PP-OCRv5_server_rec'    = model besar (lebih akurat untuk bentuk huruf mirip seperti Q/O/0, lebih lambat di CPU)
# 'en_PP-OCRv5_mobile_rec' = model kecil khusus Inggris (lebih cepat)
OCR_MODEL = 'en_PP-OCRv5_mobile_rec'
ocr = TextRecognition(model_name=OCR_MODEL, device='cpu')

def crop_plate(img, x1, y1, x2, y2, mx=0.04, my=0.10):
    h, w = img.shape[:2]
    bw, bh = x2 - x1, y2 - y1
    return img[int(max(0, y1 - bh * my)):int(min(h, y2 + bh * my)), int(max(0, x1 - bw * mx)):int(min(w, x2 + bw * mx))]

def read_plates(img, conf=0.5):
    out = []
    for x1, y1, x2, y2 in det.predict(img, imgsz=640, conf=conf, verbose=False)[0].boxes.xyxy.cpu().numpy():
        crop = crop_main_line(crop_plate(img, x1, y1, x2, y2))
        if crop.size == 0:
            continue
        r = next(iter(ocr.predict(input=crop, batch_size=1)))
        r = r.json.get('res', r.json)
        plate = fix_plate(r['rec_text'])
        out.append({'plat': format_plate(plate) if plate else r['rec_text'], 'valid': is_valid(plate),
                    'raw': r['rec_text'], 'skor': round(float(r['rec_score']), 3), 'box': (int(x1), int(y1), int(x2), int(y2))})
    return out

def show(paths, cols=4):
    rows = (len(paths) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 4 * rows))
    for k, p in enumerate(paths):
        img = cv2.imread(str(p))
        res = read_plates(img)
        for r in res:
            x1, y1, x2, y2 = r['box']; col = (0, 200, 0) if r['valid'] else (0, 0, 255)
            cv2.rectangle(img, (x1, y1), (x2, y2), col, 3)
            cv2.putText(img, r['plat'], (x1, max(30, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, col, 3)
        plt.subplot(rows, cols, k + 1); plt.imshow(img[:, :, ::-1]); plt.axis('off')
        plt.title(', '.join(r['plat'] for r in res) or 'tidak terdeteksi', fontsize=11)
        print(Path(p).name, res)
    plt.tight_layout(); plt.show()

show(random.sample(sorted(Path('/content/detect/test/images').glob('*')), 8))

### Coba dengan foto sendiri

In [ ]:
# Taruh foto kendaraan di folder Drive ini (upload lewat drive.google.com, atau drag & drop ke panel Files Colab).
# Widget files.upload() hanya jalan di tab browser Colab, tidak di VS Code / bila output dibuka ulang.
FOTO = Path(f'{ROOT}/foto_uji')
FOTO.mkdir(parents=True, exist_ok=True)
paths = sorted(p for p in FOTO.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
if paths:
    show(paths)
else:
    print(f'Belum ada foto di {FOTO} — upload foto ke folder itu lalu jalankan ulang sel ini.')

In [ ]:
# Bandingkan model OCR kecil vs besar pada foto di folder foto_uji (potongan plat yang sama untuk kedua model).
import time
MODEL_OCR = {'mobile': TextRecognition(model_name='en_PP-OCRv5_mobile_rec', device='cpu'),
             'server': ocr if OCR_MODEL == 'PP-OCRv5_server_rec' else TextRecognition(model_name='PP-OCRv5_server_rec', device='cpu')}
waktu = {k: [] for k in MODEL_OCR}
for p in paths:
    img = cv2.imread(str(p))
    for x1, y1, x2, y2 in det.predict(img, imgsz=640, conf=0.5, verbose=False)[0].boxes.xyxy.cpu().numpy():
        crop = crop_main_line(crop_plate(img, x1, y1, x2, y2))
        if crop.size == 0:
            continue
        baris = []
        for nama, m in MODEL_OCR.items():
            t = time.time()
            r = next(iter(m.predict(input=crop, batch_size=1)))
            waktu[nama].append(time.time() - t)
            r = r.json.get('res', r.json)
            baris.append(f"{nama}: {r['rec_text']!r:18} -> {format_plate(fix_plate(r['rec_text'])) or '-':12} ({float(r['rec_score']):.2f})")
        print(Path(p).name, '|', ' | '.join(baris))
print({k: f'{1000 * np.mean(v):.0f} ms/plat' for k, v in waktu.items() if v})

### Pencocokan ke plat terdaftar
OCR bisa salah membaca karakter yang bentuknya mirip (mis. `Q` terbaca `0`). Karena gerbang hanya perlu tahu plat terdaftar mana yang dimaksud,
hasil OCR dicocokkan dengan aturan: **EXACT** (sama persis) → **MIRIP** (beda hanya karakter mirip) → **EDIT** (selisih 1 karakter).
**AMBIGU** = lebih dari satu plat cocok; RFID yang menentukan.

In [ ]:
# Cara 2: cocokkan hasil OCR ke daftar plat terdaftar dengan toleransi karakter mirip (O/Q/D/0, I/1, B/8, S/5, Z/2, ...).
# Isi TERDAFTAR dengan plat yang ada di registered_vehicles (atau plat pada foto uji Anda).
TERDAFTAR = ['BM 6408 QO', 'B 1234 XYZ', 'A 1234 BC']

for p in paths:
    for r in read_plates(cv2.imread(str(p))):
        m = match_plate(r['raw'], TERDAFTAR)
        print(f"{Path(p).name:25} OCR {r['raw']!r:14} -> {m['cara']:9} {m['plat'] or '-':12} kandidat={m['kandidat']}")


## 4. Ekspor model untuk Mini PC
Mini PC menjalankan ALPR dengan **onnxruntime / OpenVINO** saja (tanpa PyTorch/PaddlePaddle — PaddlePaddle belum ada untuk Python 3.14).
Sel ini membuat 3 file di `models/minipc_yolo26n/` (Drive):

| File | Isi |
|---|---|
| `plate_det.onnx` | YOLO26n detektor plat (input 640×640, end-to-end: keluaran `[1, 300, 6]` = x1, y1, x2, y2, skor, kelas) |
| `plate_rec.onnx` | PaddleOCR `en_PP-OCRv5_mobile_rec` |
| `plate_rec_dict.txt` | daftar karakter model OCR |

Unduh ketiganya ke folder `alpr_model/weights/` (atau folder lain lalu set `config.ALPR_MODEL_DIR`) di Mini PC.
`smart_gate/vision/local_alpr.py` mengenali format keluaran secara otomatis (YOLOv8 `[1, 5, N]` + NMS, atau YOLO26 `[1, 300, 6]` tanpa NMS).
Sel berikutnya membandingkan hasil ONNX dengan model asli agar konversi terbukti tidak mengubah hasil.


In [ ]:
!pip install -q onnxruntime onnx onnxslim
import yaml
EXPORT = Path(f'{MODELS}/minipc_yolo26n')
EXPORT.mkdir(parents=True, exist_ok=True)

# 1) YOLO26n -> ONNX (ukuran input tetap 640; end-to-end, NMS tidak diperlukan)
onnx_det = YOLO(BEST).export(format='onnx', imgsz=640, opset=17, simplify=True, dynamic=False)
shutil.copy(onnx_det, EXPORT / 'plate_det.onnx')
import onnxruntime as ort
_o = ort.InferenceSession(str(EXPORT / 'plate_det.onnx'), providers=['CPUExecutionProvider']).get_outputs()[0]
print('Keluaran plate_det.onnx:', _o.name, _o.shape)   # YOLO26 end-to-end: [1, 300, 6]

# 2) PaddleOCR rec -> ONNX (model diunduh PaddleOCR ke ~/.paddlex/official_models/)
REC_NAME = 'en_PP-OCRv5_mobile_rec'
TextRecognition(model_name=REC_NAME, device='cpu')          # pastikan model sudah terunduh
rec_dir = next(p.parent for p in Path.home().rglob(f'{REC_NAME}/inference.yml'))

# paddle2onnx 2.x belum ada untuk Python 3.13 (Python Colab), jadi konversi dilakukan di venv Python 3.12 terpisah.
# --seed = venv ikut berisi pip/setuptools (dibutuhkan saat import paddle).
!pip install -q uv
!uv venv -q --seed --clear --python 3.12 /content/p2o   # --clear: timpa venv lama tanpa bertanya
!uv pip install -q --python /content/p2o/bin/python paddlepaddle==3.2.0 paddle2onnx==2.1.0 packaging onnx setuptools
!rm -rf /content/rec_onnx && mkdir -p /content/rec_onnx
!/content/p2o/bin/paddle2onnx --model_dir "{rec_dir}" --model_filename inference.json --params_filename inference.pdiparams --save_file /content/rec_onnx/inference.onnx --opset_version 14
if not Path('/content/rec_onnx/inference.onnx').exists():
    raise RuntimeError('Konversi PaddleOCR -> ONNX gagal, lihat pesan paddle2onnx di atas.')
shutil.copy('/content/rec_onnx/inference.onnx', EXPORT / 'plate_rec.onnx')

# 3) Kamus karakter (urutan HARUS sama dengan model; indeks 0 = blank CTC ditambahkan di Mini PC)
chars = yaml.safe_load(open(rec_dir / 'inference.yml'))['PostProcess']['character_dict']
(EXPORT / 'plate_rec_dict.txt').write_text('\n'.join(chars), encoding='utf-8')

for f in sorted(EXPORT.iterdir()):
    print(f'{f.name:22} {f.stat().st_size / 1e6:6.1f} MB')

In [ ]:
# Bandingkan ONNX vs model asli pada gambar test (deteksi: jumlah & posisi kotak, OCR: teks pada potongan yang sama)
import onnxruntime as ort, math
det_onnx = YOLO(str(EXPORT / 'plate_det.onnx'), task='detect')
rec_sess = ort.InferenceSession(str(EXPORT / 'plate_rec.onnx'), providers=['CPUExecutionProvider'])
CH = ['blank'] + (EXPORT / 'plate_rec_dict.txt').read_text(encoding='utf-8').split('\n')

def rec_onnx(crop):        # prapemrosesan & decode sama dengan smart_gate/vision/local_alpr.py
    h, w = crop.shape[:2]; W = max(320, int(48 * w / h)); rw = min(W, math.ceil(48 * w / h))
    x = (cv2.resize(crop, (rw, 48)).astype(np.float32).transpose(2, 0, 1) / 255 - 0.5) / 0.5
    b = np.zeros((1, 3, 48, W), np.float32); b[0, :, :, :rw] = x
    p = rec_sess.run(None, {rec_sess.get_inputs()[0].name: b})[0][0]
    ch = CH + [' '] * (p.shape[1] - len(CH)); out, prev = [], -1
    for i in p.argmax(1):
        if i != prev and i != 0: out.append(ch[i])
        prev = i
    return ''.join(out)

ocr_pd = TextRecognition(model_name=REC_NAME, device='cpu')
sama_det = sama_ocr = n_det = n_ocr = 0
for p in sorted(Path('/content/detect/test/images').glob('*'))[:60]:
    img = cv2.imread(str(p))
    a = det.predict(img, imgsz=640, conf=0.5, verbose=False)[0].boxes.xyxy.cpu().numpy()
    b = det_onnx.predict(img, imgsz=640, conf=0.5, verbose=False)[0].boxes.xyxy.cpu().numpy()
    n_det += 1; sama_det += len(a) == len(b) and (len(a) == 0 or np.abs(a - b).max() < 3)
    for x1, y1, x2, y2 in a:
        crop = crop_main_line(crop_plate(img, x1, y1, x2, y2))
        if crop.size == 0: continue
        r = next(iter(ocr_pd.predict(input=crop, batch_size=1))); t_pd = r.json.get('res', r.json)['rec_text']
        n_ocr += 1; sama_ocr += plate_utils.clean(t_pd) == plate_utils.clean(rec_onnx(crop))
print(f'Deteksi ONNX sama dengan .pt : {sama_det}/{n_det} gambar')
print(f'OCR ONNX sama dengan Paddle  : {sama_ocr}/{n_ocr} plat')

### Ukuran input YOLO lebih kecil (lebih cepat di Mini PC)
Waktu deteksi kira-kira sebanding dengan jumlah piksel: 416 ≈ 2,4× dan 320 ≈ 4× lebih cepat dari 640,
tetapi plat kecil/jauh lebih sulit terdeteksi. Sel ini menghitung mAP50 & recall test di tiap ukuran dan mengekspor
`plate_det_<ukuran>.onnx` ke `models/minipc_yolo26n/`. Pilih ukuran terkecil yang recall-nya masih hampir sama dengan 640,
lalu di Mini PC ganti nama file itu menjadi `plate_det.onnx` (kode membaca ukuran input dari file model).


In [ ]:
UKURAN = [640, 416, 320]
model = YOLO(BEST)
for sz in UKURAN:
    m = model.val(data='/content/plate.yaml', split='test', imgsz=sz, verbose=False, plots=False)
    print(f'{sz}px | mAP50 = {m.box.map50:.3f} | recall = {m.box.mr:.3f} | precision = {m.box.mp:.3f}')
    if sz != 640:
        f = model.export(format='onnx', imgsz=sz, opset=17, simplify=True, dynamic=False)
        shutil.copy(f, EXPORT / f'plate_det_{sz}.onnx')
for f in sorted(EXPORT.glob('plate_det*.onnx')):
    print(f'{f.name:22} {f.stat().st_size / 1e6:6.1f} MB')